# 03 — Cross-Vocabulary Translation

A common holonic workflow: one holon holds data in a source vocabulary
(Common Core Ontology, domain standard, internal schema); another
holon expects a target vocabulary (Schema.org for web publishing,
internal canonical form). A portal carries the CONSTRUCT that
translates between them.

This notebook shows the pattern end-to-end with a stylized subset of
CCO classes and predicates mapped to Schema.org equivalents.


## Setup — a source holon with CCO-flavored data


In [ ]:
from holonic import HolonicDataset

ds = HolonicDataset()

ds.add_holon("urn:holon:cco-source", "CCO Source")
ds.add_interior("urn:holon:cco-source", '''
    @prefix cco: <http://www.ontologyrepository.com/CommonCoreOntologies/> .

    <urn:agent:alice> a cco:Person ;
        cco:has_text_value "Alice Morgenstern" ;
        cco:is_affiliated_with <urn:org:acme> .

    <urn:org:acme> a cco:CommercialOrganization ;
        cco:has_text_value "Acme Corp" ;
        cco:is_located_in <urn:place:nyc> .

    <urn:place:nyc> a cco:City ;
        cco:has_text_value "New York City" .
''')


## Target holon — will receive Schema.org-flavored translations


In [ ]:
ds.add_holon("urn:holon:schema-target", "Schema.org Target")
ds.add_boundary("urn:holon:schema-target", '''
    @prefix schema: <https://schema.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .

    <urn:shapes:PersonShape> a sh:NodeShape ;
        sh:targetClass schema:Person ;
        sh:property [
            sh:path schema:name ;
            sh:minCount 1 ;
            sh:datatype xsd:string ;
            sh:severity sh:Violation
        ] .
''')


## Register the translation portal

The CONSTRUCT mechanically maps CCO → Schema.org triples. In practice
this kind of alignment may be factored into an alignment holon (see
`cga:AlignmentHolon` in the ontology) referenced by the portal via
`cga:usesAlignment`.


In [ ]:
construct = '''
    PREFIX cco: <http://www.ontologyrepository.com/CommonCoreOntologies/>
    PREFIX schema: <https://schema.org/>

    CONSTRUCT {
        ?person a schema:Person ;
            schema:name ?pname ;
            schema:affiliation ?org .
        ?org a schema:Organization ;
            schema:name ?oname ;
            schema:location ?place .
        ?place a schema:Place ;
            schema:name ?placename .
    }
    WHERE {
        ?person a cco:Person ;
            cco:has_text_value ?pname ;
            cco:is_affiliated_with ?org .
        ?org a cco:CommercialOrganization ;
            cco:has_text_value ?oname ;
            cco:is_located_in ?place .
        ?place a cco:City ;
            cco:has_text_value ?placename .
    }
'''

ds.add_portal(
    "urn:portal:cco-to-schemaorg",
    source_iri="urn:holon:cco-source",
    target_iri="urn:holon:schema-target",
    construct_query=construct,
    label="CCO → Schema.org",
)


## Traverse with validation

The target's boundary shapes enforce that `schema:Person` instances
have a name. If the CONSTRUCT had dropped names, the membrane would
report `Compromised` and the data would NOT be injected.


In [ ]:
projected, membrane = ds.traverse(
    "urn:holon:cco-source",
    "urn:holon:schema-target",
    validate=True,
    agent_iri="urn:agent:translator",
)
print(f"injected triples: {len(projected)}")
print(f"membrane health:  {membrane.health}")


## Verify the target now holds Schema.org data


In [ ]:
rows = ds.query('''
    PREFIX schema: <https://schema.org/>
    SELECT ?person ?name ?org
    WHERE {
        GRAPH <urn:holon:schema-target/interior> {
            ?person a schema:Person ;
                schema:name ?name ;
                schema:affiliation ?org .
        }
    }
''')
rows


## Where to go next

- `02_portal_traversal` — foundational portal mechanics
- `09_projection_plugins` — declaring reusable transformation pipelines
  as first-class RDF specs
